## Exploratory Data Analysis to Support Housing Price Research

**Author**: Baher Alabbar  
**Date**: February 2025  


### Objective of the EDA 

This notebook aims to conduct a thorough exploratory data analysis (EDA) on the House Price dataset with the following goals:

1. Examine individual features and their relationships with the target variable.
2. Assess data quality by identifying missing values, noisy features, and variables that may require transformation or removal.
3. Identify opportunities for meaningful feature engineering to improve predictive performance.
4. Generate insights that inform model selection and evaluation.

---

### Target Variable

The target variable is `SalePrice`, measured in U.S. dollars. Due to its highly right-skewed distribution, we primarily work with its `log1p` transformation, denoted as `log_sale_price`. Applying `log1p` reduces the influence of extreme values and helps stabilize variance, which can improve model performance. Model predictions are generated in the log-transformed space and subsequently transformed back to the original dollar scale using the inverse transformation (`expm1`) for interpretation and evaluation.

---
### EDA Structure

Below is an outline of the EDA sections to guide navigation and ensure coverage of key areas.

<details>
<summary><strong>Click to expand the EDA Sections</strong></summary>

- [Load and Inspect the Data](#load-and-inspect-the-data)  
  Load the dataset, check its shape, structure, and basic statistics.

- [1. Dataset Feature Descriptions](#1-columns-and-descriptions)  
  Explanation of each feature in the dataset.

- [2. Data Cleaning and Preprocessing](#2-data-cleaning-and-preprocessing)  
  Handle missing values, fix data types, and prepare for analysis.

- [3. Outlier Detection and Target Variable Transformation](#3-outlier-detection-and-target-variable-transformation)  
  Visualize and understand outliers and whether log1p is needed.

- [4. Categorical Feature Analysis](#4-categorical-feature-analysis)   
  Analyze categorical features and their effect on sale prices.

- [5. Numerical Feature Analysis](#5-numerical-feature-analysis)   
  Analyze numerical features and their effect on sale prices.

- [6. Domain-Specific Feature Analysis](#6-domain-specific-feature-analysis)
  Examine how housing-related features such as neighborhood, zoning, lot characteristics, and structural attributes influence sale prices.

- [7. Domain-Specific Temporal Feature Analysis](#7-domain-specific-temporal-feature-analysis)
  Analyze how time-related features, including year built, year remodeled, month sold, and year sold, affect housing prices.

- [8. Relationships Between Features](#8-relationships-between-features)  
  Examine how features interact or correlate with one another.

- [9. Feature Engineering](#9-feature-engineering)  
  Create, evaluate, and justify new features that may improve the model.

- [10. Summary and Next Steps](#10-summary-and-next-steps)  
  Wrap up findings, hypotheses, and prepare for modeling.

- [Acknowledgments](#acknowledgments)

</details>

### Load and Inspect The Data

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline

In [3]:
pd.options.display.max_columns = None  # Show all columns
pd.options.display.max_rows = 500

import warnings

# ignore FutureWarnings and UserWarnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [4]:
df = pd.read_csv("../data/split/train.csv")
df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,255,20,RL,70.0,8400,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Norm,Norm,1Fam,1Story,5,6,1957,1957,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,Gd,CBlock,TA,TA,No,Rec,922,Unf,0,392,1314,GasA,TA,Y,SBrkr,1314,0,0,1314,1,0,1,0,3,1,TA,5,Typ,0,NaN,Attchd,1957.0,RFn,1,294,TA,TA,Y,250,0,0,0,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal,145000
1,1067,60,RL,59.0,7837,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,6,7,1993,1994,Gable,CompShg,VinylSd,VinylSd,NaN,0.0,Gd,TA,PConc,Gd,TA,No,Unf,0,Unf,0,799,799,GasA,Gd,Y,SBrkr,799,772,0,1571,0,0,2,1,3,1,TA,7,Typ,1,TA,Attchd,1993.0,RFn,2,380,TA,TA,Y,0,40,0,0,0,0,NaN,NaN,NaN,0,5,2009,WD,Normal,178000
2,639,30,RL,67.0,8777,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,Edwards,Feedr,Norm,1Fam,1Story,5,7,1910,1950,Gable,CompShg,MetalSd,Wd Sdng,NaN,0.0,TA,TA,CBlock,Fa,TA,No,Unf,0,Unf,0,796,796,GasA,Gd,Y,FuseA,796,0,0,796,0,0,1,0,2,1,TA,4,Typ,0,NaN,NaN,NaN,NaN,0,0,NaN,NaN,P,328,0,164,0,0,0,NaN,MnPrv,NaN,0,5,2008,WD,Normal,85000
3,800,50,RL,60.0,7200,Pave,NaN,Reg,Lvl,AllPub,Corner,Gtl,SWISU,Feedr,Norm,1Fam,1.5Fin,5,7,1937,1950,Gable,CompShg,Wd Sdng,Wd Sdng,BrkFace,252.0,TA,TA,BrkTil,Gd,TA,No,ALQ,569,Unf,0,162,731,GasA,Ex,Y,SBrkr,981,787,0,1768,1,0,1,1,3,1,Gd,7,Typ,2,TA,Detchd,1939.0,Unf,1,240,TA,TA,Y,0,0,264,0,0,0,NaN,MnPrv,NaN,0,6,2007,WD,Normal,175000
4,381,50,RL,50.0,5000,Pave,Pave,Reg,Lvl,AllPub,Inside,Gtl,SWISU,Norm,Norm,1Fam,1.5Fin,5,6,1924,1950,Gable,CompShg,BrkFace,Wd Sdng,NaN,0.0,TA,TA,BrkTil,TA,TA,No,LwQ,218,Unf,0,808,1026,GasA,TA,Y,SBrkr,1026,665,0,1691,0,0,2,0,3,1,Gd,6,Typ,1,Gd,Detchd,1924.0,Unf,1,308,TA,TA,Y,0,0,242,0,0,0,NaN,NaN,NaN,0,5,2010,WD,Normal,127000


In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Id,1168.0,730.904966,425.369088,1.0,360.75,732.5,1101.75,1460.0
MSSubClass,1168.0,56.849315,42.531862,20.0,20.00,50.0,70.00,190.0
LotFrontage,951.0,70.343849,24.897021,21.0,59.00,70.0,80.00,313.0
LotArea,1168.0,10689.642123,10759.366198,1300.0,7587.25,9600.0,11700.00,215245.0
OverallQual,1168.0,6.121575,1.367619,1.0,5.00,6.0,7.00,10.0
OverallCond,1168.0,5.584760,1.116062,1.0,5.00,5.0,6.00,9.0
YearBuilt,1168.0,1970.965753,30.675495,1872.0,1953.00,1972.0,2001.00,2010.0
YearRemodAdd,1168.0,1984.897260,20.733955,1950.0,1966.00,1994.0,2004.00,2010.0
MasVnrArea,1162.0,103.771945,173.032238,0.0,0.00,0.0,166.00,1378.0
BsmtFinSF1,1168.0,446.023973,459.070977,0.0,0.00,384.5,721.00,5644.0


In [6]:
df.shape

(1168, 81)

It is clear that the dataset is high-dimensional, containing 81 features, making a detailed investigation of every variable impractical. Therefore, an initial feature screening step is necessary to reduce the features amount and focus the analysis on variables that are most relevant and informative for further investigation.

### 1. Columns and Descriptions

In [12]:
df.columns

Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
       'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1',
       'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
       'GarageCond', 'PavedDrive

### Feature Overview (Grouped Summary)

**Identifiers & Target**
- **Id**: Unique identifier for each property.
- **SalePrice**: Final sale price of the house (target variable).

**Property Type & Zoning**
- **MSSubClass**: Dwelling type and style category.
- **MSZoning**: General zoning classification of the property.
- **BldgType**: Type of building structure (e.g., single-family, townhouse).
- **HouseStyle**: Architectural style of the house.

**Lot & Land Characteristics**
- **LotFrontage**: Linear feet of street connected to the property.
- **LotArea**: Total lot size in square feet.
- **LotShape**: Shape regularity of the lot.
- **LandContour**: Flatness or slope of the land.
- **LotConfig**: Lot position and configuration.
- **LandSlope**: Slope steepness of the property.
- **Street**: Type of road access.
- **Alley**: Type of alley access, if any.
- **Utilities**: Available utilities for the property.

**Location & Neighborhood**
- **Neighborhood**: Physical neighborhood within Ames city limits.
- **Condition1**: Proximity to roads, railroads, or amenities.
- **Condition2**: Secondary proximity condition, if applicable.

**Overall Quality & Condition**
- **OverallQual**: Overall material and finish quality.
- **OverallCond**: Overall condition of the house.

**Construction & Renovation**
- **YearBuilt**: Original construction year.
- **YearRemodAdd**: Year of last remodel or addition.
- **RoofStyle**: Roof shape or design.
- **RoofMatl**: Roof material.
- **Foundation**: Foundation type.

**Exterior Features**
- **Exterior1st**: Primary exterior covering material.
- **Exterior2nd**: Secondary exterior covering material.
- **MasVnrType**: Masonry veneer type.
- **MasVnrArea**: Masonry veneer area in square feet.
- **ExterQual**: Exterior material quality.
- **ExterCond**: Exterior material condition.

**Basement Features**
- **BsmtQual**: Basement height quality.
- **BsmtCond**: Basement condition.
- **BsmtExposure**: Walkout or garden-level exposure.
- **BsmtFinType1**: Quality of main finished basement area.
- **BsmtFinSF1**: Finished basement square feet (type 1).
- **BsmtFinType2**: Quality of secondary finished basement area.
- **BsmtFinSF2**: Finished basement square feet (type 2).
- **BsmtUnfSF**: Unfinished basement area.
- **TotalBsmtSF**: Total basement area.
- **BsmtFullBath**: Basement full bathrooms.
- **BsmtHalfBath**: Basement half bathrooms.

**Heating, Cooling & Utilities**
- **Heating**: Heating system type.
- **HeatingQC**: Heating quality and condition.
- **CentralAir**: Presence of central air conditioning.
- **Electrical**: Electrical system type.

**Living Area & Room Counts**
- **1stFlrSF**: First-floor living area.
- **2ndFlrSF**: Second-floor living area.
- **LowQualFinSF**: Low-quality finished living area.
- **GrLivArea**: Total above-ground living area.
- **FullBath**: Full bathrooms above grade.
- **HalfBath**: Half bathrooms above grade.
- **BedroomAbvGr**: Bedrooms above grade.
- **KitchenAbvGr**: Kitchens above grade.
- **KitchenQual**: Kitchen quality.
- **TotRmsAbvGrd**: Total rooms above grade (excluding bathrooms).
- **Functional**: Overall home functionality rating.
- **Fireplaces**: Number of fireplaces.
- **FireplaceQu**: Fireplace quality.

**Garage Features**
- **GarageType**: Garage location/type.
- **GarageYrBlt**: Garage construction year.
- **GarageFinish**: Garage interior finish.
- **GarageCars**: Garage capacity (number of cars).
- **GarageArea**: Garage size in square feet.
- **GarageQual**: Garage quality.
- **GarageCond**: Garage condition.
- **PavedDrive**: Driveway paving type.

**Porch, Deck & Outdoor Amenities**
- **WoodDeckSF**: Wood deck area.
- **OpenPorchSF**: Open porch area.
- **EnclosedPorch**: Enclosed porch area.
- **3SsnPorch**: Three-season porch area.
- **ScreenPorch**: Screen porch area.
- **PoolArea**: Pool area in square feet.
- **PoolQC**: Pool quality.
- **Fence**: Fence quality.
- **MiscFeature**: Miscellaneous features not covered elsewhere.
- **MiscVal**: Value of miscellaneous features.

**Sale Information**
- **MoSold**: Month of sale.
- **YrSold**: Year of sale.
- **SaleType**: Type of sale transaction.
- **SaleCondition**: Condition under which the sale occurred.

*This summary is inspired by the official feature descriptions provided with the dataset.*

**Next step:** initial data processing and feature screening will be performed to reduce dimensionality and focus on the most informative variables.

### 2. Data Cleaning and Preprocessing

#### 2.1 Handling Missing Values